# Per-Tensor and Per-Channel Quantization

###  Import Required Libraries

In [24]:
import torch
import numpy as np
import pandas as pd

### Part A: Per-Tensor Symmetric Quantization

In [2]:
def per_tensor_quantize(tensor):
    scale  =  torch.abs(tensor).max().item() / 127
    quantized = torch.round(tensor/scale)
    quantized = quantized.clamp(-127,127).to(torch.int8)
    dequantized = quantized.float() * scale
    return quantized, dequantized, scale

### Part B: Per-Output-Channel Symmetric Quantization 

In [17]:
def per_channel_quantize(tensor):
    out_channels = tensor.shape[0]
    scale = torch.zeros(out_channels)
    for index in range(out_channels):
        sub_tensor = tensor.select(0,index)
        scale[index] = sub_tensor.abs().max().item() / 127
    scale_shape = [1] * tensor.dim()
    scale_shape[0] = -1
    scale = scale.view(scale_shape)
    quantized = torch.round(tensor/scale)
    quantized = quantized.clamp(-127,127).to(torch.int8)
    dequantized = quantized.float() * scale
    return quantized, dequantized, scale

### Input

In [12]:
np.random.seed(42)
conv_weights = np.random.randn(8,3,3,3).astype(np.float32)
conv_weights = torch.tensor(conv_weights)
# Create range imbalance across output channels
conv_weights[0] *= 0.1    # very small range
conv_weights[1] *= 0.5    # small range
conv_weights[6] *= 5.0    # large range
conv_weights[7] *= 10.0   # very large range

print("Shape:", conv_weights.shape)

Shape: torch.Size([8, 3, 3, 3])


### Apply Per-Tensor Symmetric Quantization

In [5]:
per_tensor_symm_q, per_tensor_symm_dq, per_tensor_scale = per_tensor_quantize(conv_weights)

### Apply Per-Output-Channel Symmetric Quantization

In [21]:
per_channel_symm_q, per_channel_symm_dq, per_channel_scale = per_channel_quantize(conv_weights)

### Compute Error

In [31]:
rows = []
for c in range(conv_weights.shape[0]):
    tensor = conv_weights[c]
    per_tensor_mae = (tensor - per_tensor_symm_q).abs().mean()
    per_channel_mae = (tensor - per_channel_symm_q).abs().mean()
    rows.append({
        "Channel": c,
        "Range (min,max)": f"({tensor.min():.4f}, {tensor.max():.4f})",
        "Per-Tensor Scale": per_tensor_scale,
        "Per-Tensor MAE": per_tensor_mae,
        "Per-Ch Scale": per_channel_scale[c],
        "Per-Ch MAE": per_channel_mae,
        "Better": "Per-Channel" if per_channel_mae < per_tensor_mae else "Per-Tensor"
    })

In [32]:
df = pd.DataFrame(rows)
avg_row = {
    "Channel": "Average",
    "Range (min,max)": "-",
    "Per-Tensor Scale": per_tensor_scale,
    "Per-Tensor MAE": df["Per-Tensor MAE"].mean(),
    "Per-Ch Scale": df["Per-Ch Scale"].mean(),
    "Per-Ch MAE": df["Per-Ch MAE"].mean(),
    "Better": "-"
}
df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)

### Output

In [33]:
df

,Channel,"Range (min,max)",Per-Tensor Scale,Per-Tensor MAE,Per-Ch Scale,Per-Ch MAE,Better
0,0,"(-0.1913, 0.1579)",0.303365,tensor(6.3537),[[[tensor(0.0015)]]],tensor(41.6684),Per-Tensor
1,1,"(-0.9798, 0.9261)",0.303365,tensor(6.3999),[[[tensor(0.0077)]]],tensor(41.6507),Per-Tensor
2,2,"(-2.6197, 1.5646)",0.303365,tensor(6.3773),[[[tensor(0.0206)]]],tensor(41.5803),Per-Tensor
3,3,"(-1.4635, 1.8862)",0.303365,tensor(6.4500),[[[tensor(0.0149)]]],tensor(41.6994),Per-Tensor
4,4,"(-1.9188, 2.4632)",0.303365,tensor(6.4172),[[[tensor(0.0194)]]],tensor(41.5581),Per-Tensor
5,5,"(-1.6075, 1.8658)",0.303365,tensor(6.3889),[[[tensor(0.0147)]]],tensor(41.5756),Per-Tensor
6,6,"(-5.3545, 13.6008)",0.303365,tensor(7.4915),[[[tensor(0.1071)]]],tensor(41.1573),Per-Tensor
7,7,"(-15.1485, 38.5273)",0.303365,tensor(10.6737),[[[tensor(0.3034)]]],tensor(42.1561),Per-Tensor
8,Average,-,0.303365,7.069028,0.061156,41.630737,-
